# Customizing Visualizations

In [3]:
import altair as alt
import pandas as pd
from vega_datasets import data
cars = data.cars.url
iowa = data.iowa_electricity.url

## Adjusting the color and transparency

In [4]:
chart = alt.Chart(cars).mark_point(opacity=0.2, color='red').encode( # Local Config
    x='Acceleration:Q',
    y='Horsepower:Q'
)
chart

alt.Chart(...)

It is possible to set chart properties via the encoding channel. Rather than mapping a property to a data column, you can map a property directly to a value using the value() function:

In [5]:
alt.Chart(cars).mark_point().encode(
    x='Acceleration:Q',
    y='Horsepower:Q',
    opacity=alt.value(0.2),
    color=alt.value('red')
)

alt.Chart(...)

## Adjusting Chart Size

The size of charts can be adjusted using the width and height properties. For example:

In [6]:
alt.Chart(cars).mark_point().encode(
    x='Acceleration:Q',
    y='Horsepower:Q'
).properties(
    width=100,
    height=150
)

alt.Chart(...)

## Customizing Colors

Altair chooses a suitable default color scheme based on the type of the data that the color encodes. These defaults can be customized using the scale() method of the Color class.

### Color Schemes
Altair includes a set of named color schemes for both categorical and sequential data, defined by the vega project; see the [Vega documentation](https://vega.github.io/vega/docs/schemes/) for a full gallery of available color schemes. These schemes can be passed to the scheme argument of the scale() method:

In [7]:
alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.Color('Acceleration:Q').scale(scheme="magma") # we can use the reverse parameter to reverse the color scheme: scale(scheme="magma", reverse=True)
)

alt.Chart(...)

### Color Domain and Range

To create a custom color scales, we can use the domain and range parameters of the scale method for the values and colors, respectively.

In [8]:
domain = ['Europe', "Japan", "USA"]
range_ = ['seagreen', 'firebrick', 'rebeccapurple']

alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.Color('Origin:N').scale(domain=domain, range=range_)
)

alt.Chart(...)

## Adjusting the Title

By default an Altair chart does not have a title. You can add a simple title by passing the title keyword argument with the data.

In [9]:
alt.Chart(cars, title="Correlation between Acceleration and Horsepower").mark_point().encode(
    x='Acceleration:Q',
    y='Horsepower:Q',
    opacity=alt.value(0.2),
    color=alt.value('red')
)

alt.Chart(...)

It is also possible to add a subtitle by passing in an alt.Title object.

In [10]:
alt.Chart(cars, title=alt.Title(
       "Car data set",
       subtitle="Negative Correlation between Acceleration and Horsepower"
   )).mark_point().encode(
    x='Acceleration:Q',
    y='Horsepower:Q'
)

alt.Chart(...)

The subtitle can run to two lines by passing a list where each list item is a line (if you don’t want to create this list manually as in the example below, you can use the wrap function from the textwrap library to split a string into a list of substrings of a certain length).

In [11]:
alt.Chart(cars, title=alt.Title(
       "Car data set",
       subtitle=["Negative Correlation", "between Acceleration and Horsepower"]
   )).mark_point().encode(
    x='Acceleration:Q',
    y='Horsepower:Q'
)

alt.Chart(...)

The Title object can also configure a number of other attributes, e.g., to anchor it to the 'start' (left) of the chart, and to orient it at the 'bottom' of the chart.

In [12]:
alt.Chart(
   iowa,
   title=alt.Title(
       "Iowa's green energy boom",
       subtitle="A growing share of the state's energy has come from renewable sources",
       anchor='middle', # 'start', 'end'
       orient='top', # 'bottom'
       offset=20
   )
).mark_area().encode(
    x="year:T",
    y=alt.Y("net_generation:Q").stack("normalize"),
    color="source:N"
)

alt.Chart(...)

## Rearranging / sorting

In [13]:
import altair as alt

# Define your order once (from bottom to top)
stack_order = ["Nuclear Energy", "Fossil Fuels", "Renewables"]

alt.Chart(
    iowa,
    title=alt.Title(
        "Iowa's green energy boom",
        subtitle="A growing share of the state's energy has come from renewable sources",
        anchor='start',
        orient='bottom',
        offset=20
    )
).transform_calculate(
    order_idx=f"indexof({stack_order}, datum.source)"
).mark_area().encode(
    x="year:T",
    y=alt.Y("net_generation:Q").stack("normalize"),
    # Reverse the list so the top-to-bottom legend matches the bottom-to-top stack
    color=alt.Color("source:N", sort=alt.EncodingSortField(field="order_idx", order="descending")),
    order=alt.Order("order_idx:Q")
)


alt.Chart(...)

In [14]:
import altair as alt
from vega_datasets import data

iowa = data.iowa_electricity()

legend_order = ["Nuclear Energy", "Renewables", "Fossil Fuels"]

alt.Chart(
    iowa,
    title=alt.Title(
        "Iowa's green energy boom",
        subtitle="A growing share of the state's energy has come from renewable sources",
        anchor="start",
        orient="bottom",
        offset=20
    )
).transform_calculate(
    order=f"indexof({legend_order}, datum.source)"
).mark_area().encode(
    x="year:T",
    y=alt.Y("net_generation:Q").stack("normalize"),
    color=alt.Color("source:N", sort=legend_order),
    order=alt.Order("order:Q", sort="descending")
)


alt.Chart(...)

#### or the not-so-elegant-but-practical-solution: exploit auto-sorting by clever naming
if you just want to generate the graph files and edit everything in an external tool anyways.

In [15]:
iowa = data.iowa_electricity()

iowa["source"] = iowa["source"].replace({
    "Renewables": "01 Renewables",
    "Fossil Fuels": "02 Fossil Fuels",
    "Nuclear Energy": "03 Nuclear Energy"
})

alt.Chart(
   iowa,
   title=alt.Title(
       "Iowa's green energy boom",
       subtitle="A growing share of the state's energy has come from renewable sources",
       anchor='middle', # 'start', 'end'
       orient='top', # 'bottom'
       offset=20
   )
).mark_area().encode(
    x="year:T",
    y=alt.Y("net_generation:Q").stack("normalize"),
    color="source:N"
)

alt.Chart(...)

## Adjusting Axis Limits

The default axis limit used by Altair is dependent on the type of the data. To fine-tune the axis limits beyond these defaults, you can use the scale() method of the axis encodings. For example, consider the following plot:

In [16]:
alt.Chart(cars).mark_point().encode(
    x='Acceleration:Q',
    y='Horsepower:Q'
)

alt.Chart(...)

Altair inherits from Vega-Lite the convention of always including the zero-point in quantitative axes; if you would like to turn this off, you can add the scale() method to the X encoding that specifies zero=False:

In [17]:
alt.Chart(cars).mark_point().encode(
    alt.X('Acceleration:Q').scale(zero=False),
    y='Horsepower:Q'
)

alt.Chart(...)

To specify exact axis limits, you can use the domain property of the scale:

In [18]:
alt.Chart(cars).mark_point().encode(
    alt.X('Acceleration:Q').scale(domain=(5, 25)),
    y='Horsepower:Q'
)

alt.Chart(...)

## Adjusting the Legend

A legend is added to the chart automatically when the color, shape or size arguments are passed to the encode() function. In this example we’ll use color.


In [19]:
iris = data.iris()

alt.Chart(iris).mark_point().encode(
    x='petalWidth',
    y='petalLength',
    color='species'
)

alt.Chart(...)

In this case, the legend can be customized by introducing the Color class and taking advantage of its legend() method. The shape and size arguments have their own corresponding classes.

The legend option on all of them expects a Legend object as its input, which accepts arguments to customize many aspects of its appearance. One example is to move the legend to another position with the orient argument.

In [20]:
alt.Chart(iris).mark_point().encode(
    x='petalWidth',
    y='petalLength',
    color=alt.Color('species').legend(orient="left")
)

alt.Chart(...)

Another thing you can do is set a title; in this case we can use the title() method directly as a shortcut or specify the title parameter inside the legend() method:.

In [21]:
alt.Chart(iris).mark_point().encode(
    x='petalWidth',
    y='petalLength',
    color=alt.Color('species').title("Species by color")
)

alt.Chart(...)

## Removing the Chart Border

Basic Altair charts are drawn with both a grid and an outside border. To create a chart with no border, you will need to remove them both.

First remove the grid using the configure_axis() method.

In [22]:
alt.Chart(iris).mark_point().encode(
    x='petalWidth',
    y='petalLength',
    color='species'
).configure_axis(
    grid=False
)

alt.Chart(...)

You’ll note that while the inside rules are gone, the outside border remains. Hide it by setting stroke=None inside configure_view() (strokeWidth=0 and strokeOpacity=0 also works):

In [23]:
alt.Chart(iris).mark_point().encode(
    x='petalWidth',
    y='petalLength',
    color='species',
    tooltip=["species", "petalLength", "petalWidth"]
).configure_axis(
    grid=False
).configure_view(
    stroke=None
).interactive()

alt.Chart(...)

In [24]:
# adding helper lines e.g. median, mean, regression lines etc.

import altair as alt
from vega_datasets import data

iris = data.iris()

points = alt.Chart(iris).mark_point().encode(
    x='petalWidth',
    y='petalLength',
    color='species',
    tooltip=['species', 'petalLength', 'petalWidth']
)

median_rule = alt.Chart(iris).mark_rule(
    color='black',
    strokeDash=[4, 4],
    strokeWidth=1.5
).encode(
    x='median(petalWidth):Q',
    tooltip=[alt.Tooltip('median(petalWidth):Q', title='Median Petal Width')]
)

mean_rule = alt.Chart(iris).mark_rule(
    color='green',
    strokeDash=[2, 4],
    strokeWidth=1
).encode(
    y='mean(petalLength):Q',
    tooltip=[alt.Tooltip('median(petalLength):Q', title='Mean Petal Length')]
)

regression_line = alt.Chart(iris).mark_line(
    color='#aa3333',
    strokeWidth=2
).transform_regression(
    'petalWidth', 'petalLength'
).encode(
    x='petalWidth',
    y='petalLength'
)

#Shaded bounding box around the setosa cluster
setosa_area = alt.Chart(iris).transform_filter(
    alt.datum.species == 'setosa'
).transform_aggregate(
    min_x='min(petalWidth)',
    max_x='max(petalWidth)',
    min_y='min(petalLength)',
    max_y='max(petalLength)'
).transform_calculate(
    x1='datum.min_x - 0.08',
    x2='datum.max_x + 0.08',
    y1='datum.min_y - 0.2',
    y2='datum.max_y + 0.2'
).mark_rect(
    fill='#4c78a8',
    fillOpacity=0.12,
    stroke='#4c78a8',
    strokeWidth=1.2,
    cornerRadius=8
).encode(
    x='x1:Q',
    x2='x2:Q',
    y='y1:Q',
    y2='y2:Q'
)

# Text label anchored above the setosa box
setosa_text = alt.Chart({'values': [{}]}).mark_text(
    text='Setosa Cluster',
    align='left',
    baseline='top',
    dy=-5,
    fontWeight='bold',
    color='#4c78a8'
).encode(
    x=alt.datum(0.1),
    y=alt.datum(0.6)
)

(points + median_rule + mean_rule + regression_line + setosa_area + setosa_text).configure_axis(
    grid=False
).configure_view(
    stroke=None
).interactive()



alt.LayerChart(...)

## Save the graph for further editing

In [27]:
!pip install vl-convert-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 31.2 MB/s  0:00:01 eta 0:00:01

[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [26]:
chart.save("iowa_energy.svg")

ValueError: Saving charts in 'svg' format requires the vl-convert-python package: see https://altair-viz.github.io/user_guide/saving_charts.html#png-svg-and-pdf-format